# Keeping Customers Before They Leave: A Data-Driven Retention Proposal for Company A
**GCI World 2026 September — Final Assignment (Course Project PoC)**  
**Author:** Islam Raad (`Islam-Raad`) — AI Engineering, University of Mosul

---

## 0. Overview & How to Run

### Purpose
This self-contained notebook implements the end-to-end analysis supporting our Proof-of-Concept (PoC) business proposal for **Company A**, an anonymous wireless telecommunications operator (~100,000 customers):
1. **Data Audit & Exploratory Data Analysis (EDA):** Structural checks, missingness patterns, documentation vs. data verification, and statistical robustness testing of lifecycle churn thresholds (~300-day handset age and months 10–12 tenure cliff).
2. **Leakage-Safe Preprocessing & Feature Engineering:** Domain-grounded behavioral, lifecycle, call-quality, and economic features.
3. **Predictive Modeling & Interpretation:** Stratified 5-fold cross-validation across baseline and boosted-tree models, held-out test evaluation, top-$k$ targeting lift, gain and permutation importance, partial dependence, and fairness checks.
4. **Realistic Business-Impact Model:** Label-shift re-basing from the 49.56% resampled dataset to realistic telecom churn priors (5%–10%), net value estimation in monetary units (MU), scenario analysis, break-even success rate ($s^*$), optimal targeting depth ($k^*$), and tornado sensitivity.

### Folder Structure & Auto-Detection
- The notebook automatically locates `Client.csv` and `Record.csv` by checking `./telecom/`, `../telecom/` (when run from `submission/`), and `/content/drive/MyDrive/Final Assignment/telecom/` (Google Colab).
- When run inside the project workspace, it writes summary metrics to `outputs/results.json`, tables to `outputs/tables/`, and 200-DPI `matplotlib` figures to `outputs/figures/`.

### Generative AI Disclosure
Generative AI tools (Claude and Google Antigravity / Gemini) were used as analytical and coding assistants during project development. All code was executed and verified against the raw dataset, all external market statistics were verified from primary sources, and the full methodology is documented and explained by the author.

> **So what:** Every figure and number in the 15-slide executive presentation is reproducibly generated from this single notebook with `random_state=42`.

---
## 1. Setup, Environment & Matplotlib Styling Helpers

In [1]:
import json
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from xgboost import XGBClassifier

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)

# Auto-detect DATA_DIR and OUTPUT_DIR (works from repo root, submission/, or Colab)
CANDIDATE_DATA_DIRS = [
    Path('./telecom'),
    Path('../telecom'),
    Path('/content/drive/MyDrive/Final Assignment/telecom'),
]
DATA_DIR = None
for cand in CANDIDATE_DATA_DIRS:
    if (cand / 'Client.csv').exists() and (cand / 'Record.csv').exists():
        DATA_DIR = cand.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate telecom dataset. Expected Client.csv and Record.csv in one of: '
        + ', '.join(str(p) for p in CANDIDATE_DATA_DIRS)
    )

PROJECT_ROOT = DATA_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
TAB_DIR = OUTPUT_DIR / 'tables'
MOD_DIR = OUTPUT_DIR / 'models'
for d in (OUTPUT_DIR, FIG_DIR, TAB_DIR, MOD_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'results.json'
if RESULTS_PATH.exists():
    RESULTS = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
else:
    RESULTS = {
        'data': {},
        'eda': {},
        'models': [],
        'final_model': {},
        'topk': [],
        'importance': {},
        'impact': {},
        'meta': {},
    }

# Font check & Matplotlib rcParams (PRD §10 — matplotlib only)
available_fonts = {f.name for f in fm.fontManager.ttflist}
ACTIVE_FONT = 'Open Sans' if 'Open Sans' in available_fonts else 'DejaVu Sans'

PALETTE = {
    'navy': '#0B1F3A',
    'churn': '#E4572E',     # Coral (risk / churn)
    'stayed': '#3B7DD8',    # Blue (stayed / baseline)
    'teal': '#0F8B8D',      # Teal (highlight / accent)
    'amber': '#F2A541',     # Amber (callout)
    'text': '#1F2933',      # Neutral dark text
    'slate': '#5B6B7B',     # Secondary text / arrows
    'grid': '#D9E0E7',      # Light horizontal grid
    'card': '#F5F7FA',      # Card / panel background
    'white': '#FFFFFF',
}

FIG_SIZES = {
    'full': (9.0, 3.6),
    'half': (4.4, 3.3),
    'wide_short': (9.0, 2.4),
    'square': (4.4, 4.0),
}

TEAL_CMAP = LinearSegmentedColormap.from_list(
    'white_teal', ['#FFFFFF', '#D5ECEC', '#6BBEC0', '#0F8B8D', '#084C4E']
)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Open Sans', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.labelcolor': PALETTE['text'],
    'axes.edgecolor': PALETTE['grid'],
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.color': PALETTE['slate'],
    'ytick.color': PALETTE['slate'],
    'legend.fontsize': 10,
    'legend.frameon': False,
    'lines.linewidth': 2.2,
    'lines.markersize': 5.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.dpi': 200,
    'savefig.facecolor': 'white',
})


def make_fig(size_key='full', nrows=1, ncols=1, **kwargs):
    """Create a matplotlib figure at exact slide placement size with constrained_layout."""
    figsize = FIG_SIZES.get(size_key, size_key)
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, **kwargs
    )
    return fig, axes


def style_ax(ax, grid_axis='y', caption=None):
    """Apply clean executive styling (no top/right spines, subtle grid, optional n= caption)."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(PALETTE['slate'])
    ax.spines['bottom'].set_color(PALETTE['slate'])
    if grid_axis in ('y', 'both'):
        ax.yaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if grid_axis in ('x', 'both'):
        ax.xaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if caption:
        ax.text(
            0.99, -0.18, caption, transform=ax.transAxes,
            ha='right', va='top', fontsize=9.5, color=PALETTE['slate']
        )
    return ax


def save_slide_fig(fig, filename):
    """Save figure at 200 DPI without bbox_inches='tight' so placed dimensions stay exact."""
    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=200, facecolor='white')
    return out_path


def save_results():
    """Persist RESULTS dictionary to outputs/results.json."""
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2), encoding='utf-8')


RESULTS['meta'] = {
    'seed': SEED,
    'interpreter': sys.executable,
    'python_version': platform.python_version(),
    'active_font': ACTIVE_FONT,
    'versions': {
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit-learn': sklearn.__version__,
        'xgboost': xgb.__version__,
        'matplotlib': matplotlib.__version__,
        'scipy': scipy.__version__,
        'joblib': joblib.__version__,
        'lightgbm': lgb.__version__ if HAS_LIGHTGBM else None,
    },
}
save_results()
print(f'DATA_DIR: {DATA_DIR}')
print(f'Active font: {ACTIVE_FONT}')
print('Versions:', json.dumps(RESULTS['meta']['versions'], indent=2))

DATA_DIR: /home/islam-raad/Projects/Final Assignment/telecom
Active font: DejaVu Sans
Versions: {
  "pandas": "3.0.2",
  "numpy": "2.4.4",
  "scikit-learn": "1.8.0",
  "xgboost": "3.4.1",
  "matplotlib": "3.10.8",
  "scipy": "1.17.1",
  "joblib": "1.5.3",
  "lightgbm": "4.7.0"
}


> **So what:** Our environment, random seed (`42`), data paths, and executive `matplotlib` design system are locked in, ensuring every chart and metric is reproducible.

---
## 2. Load & Merge Dataset
*Pending T0.5 — Load Client.csv and Record.csv, verify schemas, and perform 1:1 merge on Customer_ID.*

---
## 3. Data Audit & Quality Assessment
*Pending Phase 1 (T1.1–T1.7) — Structure, missingness, negative/odd values, outliers, near-duplicates, and documentation vs. data audit.*

---
## 4. Exploratory Data Analysis & Robustness Checks
*Pending Phase 2 (T2.1–T2.11) — Device age (~300d) robustness, tenure cliff (months 10–12), entanglement, usage collapse, revenue, care calls, and fairness.*

---
## 5. Problem Definition & Decision Framing
*Pending Phase 4 (T4.1–T4.4) — Decision matrix, problem statement, target/task justification, and operational KPI.*

---
## 6. Leakage-Safe Preprocessing & Feature Engineering
*Pending Phase 5 (T5.1–T5.4) — Pipelines, domain features, leakage audit, and fairness variants.*

---
## 7. Predictive Modeling & Experiments
*Pending T0.6 baseline & Phase 6 (T6.1–T6.6) — Stratified 70/30 split, 5-fold CV experiments (M0–M6), and held-out test evaluation.*

---
## 8. Model Evaluation & Interpretation
*Pending Phase 7 (T7.1–T7.8) — Confusion matrix, top-k targeting table & lift curve, calibration, gain/permutation importance, partial dependence, ablation, and lever table.*

---
## 9. Business-Impact Model & Realism Adjustment
*Pending Phase 8 (T8.1–T8.8) — Label-shift re-basing (49.56% to 5%/10%), net impact scenarios, optimal k*, break-even s*, tornado sensitivity, and Lane 2 pilot sizing.*

---
## 10. Export Summary & Verification
*Pending Phase 9 (T9.3) — Final verification of outputs/results.json, figures, and tables.*

---
## 11. Conclusion, Limitations & Next Steps
*Pending Phase 9 — Key takeaways, analytical limitations, ethical safeguards, and 90-day pilot roadmap.*